In [1]:
import json
import base64
import requests
import tensorflow as tf

def predict_with_http_request():
    # --- Data Preparation ---
    data = {
        'age': [60.0],
        'anaemia': [0],
        'creatinine_phosphokinase': [582],
        'diabetes': [0],
        'ejection_fraction': [20],
        'high_blood_pressure': [1],
        'platelets': [265000.0],
        'serum_creatinine': [1.9],
        'serum_sodium': [130],
        'sex': [1],
        'smoking': [1],
        'time': [4]
    }
    
    print("--- MENYIAPKAN DATA ---")

    # --- Helper Functions ---
    def _float_feature(value):
        return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))

    def _int64_feature(value):
        return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

    def create_example(data_dict):
        feature = {}
        for key, value in data_dict.items():
            if isinstance(value[0], float):
                feature[key] = _float_feature(value[0])
            else:
                feature[key] = _int64_feature(value[0])
        return tf.train.Example(features=tf.train.Features(feature=feature))

    # --- Serialization & Encoding ---
    example_proto = create_example(data).SerializeToString()
    b64_string = base64.b64encode(example_proto).decode('utf-8')

    # --- TF Serving Request ---
    URL = "https://submission-mlops-production.up.railway.app/v1/models/heart_failure_model:predict"
    
    payload = {
        "signature_name": "serving_default",
        "instances": [
            {
                "examples": {"b64": b64_string}
            }
        ]
    }

    print(f"Mengirim request ke: {URL}")
    
    try:
        response = requests.post(URL, data=json.dumps(payload))
        response.raise_for_status()
        
        result = response.json()
        print("\n--- RESPON SERVER ---")
        
        # --- Response Parsing Logic (Updated) ---
        predictions = result['predictions'][0]
        
        # Check if response is a List or Dictionary to prevent errors
        if isinstance(predictions, list):
            # Format: [0.123]
            prediction_value = predictions[0]
        else:
            # Format: {"output_0": [0.123]}
            # Get the first value regardless of the key name
            prediction_value = list(predictions.values())[0][0]
        
        print(f"Prediction Value: {prediction_value:.4f}")
        
        # --- Conclusion ---
        status = "RISIKO TINGGI (Meninggal)" if prediction_value > 0.5 else "RISIKO RENDAH (Selamat)"
        print(f"Conclusion: {status}")

    except requests.exceptions.ConnectionError:
        print("\n[ERROR] Gagal terhubung ke Docker.")
        print("Pastikan Container Docker sudah berjalan.")
    except Exception as e:
        print(f"\n[ERROR] Terjadi kesalahan: {e}")
        if 'result' in locals():
            print("Isi Result:", result)

if __name__ == "__main__":
    predict_with_http_request()

--- MENYIAPKAN DATA ---
Mengirim request ke: https://submission-mlops-production.up.railway.app/v1/models/heart_failure_model:predict

--- RESPON SERVER ---
Prediction Value: 1.0000
Conclusion: RISIKO TINGGI (Meninggal)
